In [1]:
import os
import paths

os.chdir(paths.PROJECT_ROOT)

# Baseline hyperparameter search (K=400)

Companion notebook for `03c_baseline_hp_search_k400.py`. Presents that script's as the tables reproduced in the supplement. 


In [2]:
import os, sys
PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

import json
import glob
import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

BASELINE_HP_DIR = os.path.join(PROJECT_ROOT, "data", "SI_results", "baseline_hp")
K400_DIR = os.path.join(BASELINE_HP_DIR, "k400_search")
GAP_DIR = os.path.join(BASELINE_HP_DIR, "gap_persistence")

EVAL_WEIGHTS = {"Tier 1": 7, "Tier 2": 5, "DECK": 2, "CS3": 2}

def weighted_mean(eval_scores):
    return sum(EVAL_WEIGHTS[k] * eval_scores[k] for k in EVAL_WEIGHTS) / sum(EVAL_WEIGHTS.values())

### Table 1 - dense K=400 search, ranked (top 20 of 100)

Each configuration was run at 5 seeds; `mean_score`/`std_score` aggregate across
them using the search script's own per-run `score` (weighted mean of Tier 1,
Tier 2, DECK and CS3, matching `EVAL_WEIGHTS` above). `All` is shown for
reference and is not part of the ranking.


In [3]:
files = glob.glob(os.path.join(K400_DIR, "search_cfg*_seed*_result.json"))
raw_rows = [json.load(open(f)) for f in files]

df = pd.DataFrame(raw_rows)
eval_df = pd.json_normalize(df["eval_scores"]).rename(columns={"Tier 1": "Tier1", "Tier 2": "Tier2"})
df = pd.concat([df.drop(columns=["eval_scores"]), eval_df], axis=1)

df_search = df.groupby("config_idx").agg(
    lr=("lr", "first"), weight_decay=("weight_decay", "first"), K=("K", "first"),
    mean_score=("score", "mean"), std_score=("score", "std"), n_seeds=("seed", "count"),
    all_stable=("stable", "all"),
    Tier1=("Tier1", "mean"), Tier2=("Tier2", "mean"), DECK=("DECK", "mean"),
    CS3=("CS3", "mean"), All=("All", "mean"),
).reset_index().sort_values("mean_score").reset_index(drop=True)

print(f"{len(df_search)} configs total (from {len(raw_rows)} config x seed runs)")
df_search.head(20)

100 configs total (from 500 config x seed runs)


,config_idx,lr,weight_decay,K,mean_score,std_score,n_seeds,all_stable,Tier1,Tier2,DECK,CS3,All
0,93,0.048039,0.0010,400,0.072067,0.002547,5,True,0.051379,0.113714,0.055960,0.056467,0.106302
1,87,0.048331,0.0100,400,0.074411,0.001126,5,True,0.056179,0.116997,0.050050,0.056117,0.108756
2,54,0.038323,0.0030,400,0.077680,0.001608,5,True,0.057918,0.119454,0.057841,0.062249,0.112113
3,5,0.046304,0.0300,400,0.081832,0.003256,5,True,0.067411,0.126667,0.043240,0.058812,0.116618
4,70,0.045423,0.0300,400,0.082053,0.003314,5,True,0.067605,0.126808,0.043607,0.059184,0.116840
5,37,0.031604,0.0001,400,0.082423,0.001850,5,True,0.062293,0.123872,0.063267,0.068407,0.117061
6,48,0.030341,0.0030,400,0.083849,0.002226,5,True,0.064422,0.125366,0.062507,0.069392,0.118520
7,24,0.036548,0.0300,400,0.084897,0.004103,5,True,0.070122,0.128696,0.048180,0.063830,0.119724
8,42,0.053813,0.2000,400,0.086422,0.000177,5,True,0.085861,0.118350,0.031959,0.063032,0.118008
9,4,0.059932,0.2000,400,0.086542,0.000182,5,True,0.085999,0.118527,0.031907,0.063113,0.118032


### Table 2 - reference comparison

The search winner against two reference baselines: the pipeline's original
default (`lr=0.05, weight_decay=0.01`), and the winner transferred down from the
K=1600 search (`lr=0.1329, weight_decay=0.0`), which does not transfer well to
K=400. Per-seed evaluations are in
`data/SI_results/baseline_hp/gap_persistence/`; `weighted_mean` is computed
exactly as the search scores its own runs.


In [4]:
best = json.load(open(os.path.join(K400_DIR, "best_baseline_config_K400.json")))

ref_rows = []
for name, label in [("original", "Copied-over default"), ("tuned", "K=1600-transferred")]:
    gap_files = sorted(glob.glob(os.path.join(GAP_DIR, f"gap_{name}_seed*_result.json")))
    gap_rows = [json.load(open(f)) for f in gap_files]
    scores = [weighted_mean(r["eval_scores"]) for r in gap_rows]
    ref_rows.append({
        "label": label, "lr": gap_rows[0]["lr"], "weight_decay": gap_rows[0]["weight_decay"],
        "K": gap_rows[0]["K"], "mean_score": float(np.mean(scores)), "n_seeds": len(scores),
    })
ref_rows.append({
    "label": "K=400 dense-search winner", "lr": best["config"]["lr"],
    "weight_decay": best["config"]["weight_decay"], "K": best["config"]["K"],
    "mean_score": best["mean_score"], "n_seeds": best["n_seeds"],
})

df_reference = pd.DataFrame(ref_rows)
df_reference["pct_vs_default"] = (
    (df_reference["mean_score"].iloc[0] - df_reference["mean_score"]) / df_reference["mean_score"].iloc[0] * 100
)
df_reference

,label,lr,weight_decay,K,mean_score,n_seeds,pct_vs_default
0,Copied-over default,0.050000,0.010,400,0.073786,5,0.000000
1,K=1600-transferred,0.132930,0.000,400,0.126629,5,-71.615614
2,K=400 dense-search winner,0.048039,0.001,400,0.072067,5,2.330182


### Export (for direct transfer into the supplement)

In [ ]:
df_search.to_csv(os.path.join(K400_DIR, "baseline_k400_search_full_table.csv"), index=False)
df_search.head(20).to_csv(os.path.join(K400_DIR, "baseline_k400_search_top20_table.csv"), index=False)
df_reference.to_csv(os.path.join(K400_DIR, "baseline_reference_comparison_table.csv"), index=False)
print("Wrote 3 CSVs to", K400_DIR)